# RobustPrompt rDPO Dataset Generator
Generates **50 intents × 18 variants = 900 DPO training rows** for the HR/IT Helpdesk domain.

- `chosen`: fixed canonical answer (same string per intent)
- `rejected`: plausible but wrong/unstable LLM output
- 9 clean paraphrases + 9 adversarial paraphrases per intent

In [ ]:
import os
!pip install huggingface_hub -q


In [ ]:
import os
import json
import time
from huggingface_hub import InferenceClient

HF_TOKEN = <YOUR_HF_TOKEN>  # Set your Hugging Face token here
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
client = InferenceClient(token=HF_TOKEN)

INTENTS = [
    # ... your same INTENTS list, unchanged ...
]

# STEP 1: Generate paraphrase QUESTIONS only (not answers)
PARAPHRASE_PROMPT = """You are a synthetic data generator. Output ONLY valid JSON.

Original Question: "{canonical_q}"

Generate 9 paraphrases of this question. Mix of:
- Clean professional rephrasing (3)
- Casual / informal phrasing (3)  
- Adversarial: typos, slang, abbreviations, grammar errors (3)

Return ONLY JSON:
{{"paraphrases": ["...", "...", "...", "...", "...", "...", "...", "...", "..."]}}"""

# STEP 2: Baseline model system prompt (vanilla, no RAG, no fine-tuning)
BASELINE_SYSTEM = "You are an HR assistant. Answer the employee's question helpfully."

def generate_paraphrases(intent):
    """Generate 9 paraphrase questions for a given intent."""
    user_msg = PARAPHRASE_PROMPT.format(canonical_q=intent["canonical_q"])
    try:
        response = client.chat_completion(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": "You are a synthetic data generator. Output ONLY valid JSON."},
                {"role": "user", "content": user_msg}
            ],
            max_tokens=1000,
            temperature=0.9
        )
        raw_text = response.choices[0].message.content
        start = raw_text.find('{')
        end = raw_text.rfind('}') + 1
        data = json.loads(raw_text[start:end])
        print(f"  + Paraphrases generated: {len(data['paraphrases'])}")
        return data["paraphrases"]

    except Exception as e:
        print(f"  + Paraphrase generation FAILED: {e}")
        return []

def get_baseline_answer(paraphrased_q):
    """Send paraphrased question to vanilla baseline model, collect its answer as rejected."""
    try:
        response = client.chat_completion(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": BASELINE_SYSTEM},
                {"role": "user", "content": paraphrased_q}
            ],
            max_tokens=300,
            temperature=0.7  # some variance so baseline answers are unstable
        )
        return response.choices[0].message.content.strip()

    except Exception as e:
        print(f"  + Baseline answer FAILED: {e}")
        return None

# MAIN LOOP
all_results = []

print(f"Starting generation for {len(INTENTS)} intents...")

for intent in INTENTS:
    print(f"\nProcessing Intent {intent['id']}: {intent['topic']}")

    # Step 1: Generate paraphrase questions
    paraphrases = generate_paraphrases(intent)
    time.sleep(2)

    # Step 2: For each paraphrase, get baseline answer → that's the rejected
    for para_q in paraphrases:
        baseline_ans = get_baseline_answer(para_q)
        time.sleep(1.5)

        if baseline_ans:
            all_results.append({
                "intent_id": intent["id"],
                "topic": intent["topic"],
                "prompt": para_q,                    # paraphrased question
                "chosen": intent["canonical_a"],     # canonical answer is always chosen
                "rejected": baseline_ans             # baseline model's unstable answer
            })
            print(f"    + Pair added for: {para_q[:60]}...")

    # Checkpoint every 5 intents
    if intent['id'] % 5 == 0:
        with open("rdpo_checkpoint.json", "w") as f:
            json.dump(all_results, f)
        print(f"  checkpoint saved at intent {intent['id']}")

# EXPORT
with open("rdpo_dataset_final.json", "w") as f:
    dpo_format = [
        {
            "prompt": d["prompt"],
            "chosen": d["chosen"],
            "rejected": d["rejected"]
        }
        for d in all_results
    ]
    json.dump(dpo_format, f, indent=2)

print(f"\nDone! Generated {len(all_results)} total pairs.")
print(f"Each pair: paraphrased question → chosen (canonical) vs rejected (baseline answer)")

Starting Llama-3 generation for 50 intents...
Processing Intent 1: sick_leave_balance
  + clean success
  + adversarial success
Processing Intent 2: carry_forward_leave
  + clean success
  + adversarial success
Processing Intent 3: maternity_leave_duration
  + clean success
  + adversarial FAILED: Expecting ',' delimiter: line 38 column 41 (char 3098)
Processing Intent 4: paternity_leave_eligibility
  + clean success
  + adversarial success
Processing Intent 5: leave_encashment_policy
  + clean success
  + adversarial success
Processing Intent 6: emergency_leave_approval
  + clean success
  + adversarial success
Processing Intent 7: unpaid_leave_policy
  + clean success
  + adversarial FAILED: Expecting value: line 8 column 14 (char 1209)
Processing Intent 8: bereavement_leave
  + clean success
  + adversarial success
Processing Intent 9: medical_certificate_requirement
  + clean success
  + adversarial success
Processing Intent 10: leave_during_notice_period
  + clean success
  + adve